# 01 Kalibrierung

**Eingabe**: Roh-FITS (Light, Dark, Flat, Bias) — aus einem Siril-Workspace oder beliebigen Ordnern  
**Ausgabe**: Pfad zu kalibrierten FITS-Einzelbildern für Notebook 02

### Ablauf
1. Siril-Workspace wählen → Unterordner werden automatisch erkannt
2. Kalibrierung: **Siril** (GUI oder Skript, empfohlen) oder **ccdproc** (Python)
3. Pfad der kalibrierten Bilder für Notebook 02 festlegen

Siril-Standardstruktur: `lights/`, `darks/`, `flats/`, `biases/` (Groß-/Kleinschreibung egal).

## Vorbedingungen prüfen

Diese Zelle prüft alle Abhängigkeiten bevor das Notebook gestartet wird:
Python-Pakete, optionale externe Programme und den Kernel-Pfad.

In [ ]:
import sys, shutil, importlib
from IPython.display import display, HTML

_checks = []

def _chk(label, ok, detail="", optional=False):
    _checks.append((label, ok, detail, optional))

# --- Python-Pakete ---
_PACKAGES = [
    ("ipyfilechooser", "ipyfilechooser"),
    ("ipywidgets",     "ipywidgets"),
    ("astropy",        "astropy"),
    ("numpy",          "numpy"),
    ("matplotlib",     "matplotlib"),
    ("astroquery",     "astroquery"),
]
for name, mod in _PACKAGES:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        _chk(f"pkg: {name}", True, ver)
    except ImportError:
        _chk(f"pkg: {name}", False, "nicht installiert — pip install " + name)

# --- Externe Programme (optional) ---
_BINS = [
    ("ASTAP",  ["/Applications/ASTAP.app/Contents/MacOS/astap"]),
    ("Siril",  ["/Applications/Siril.app/Contents/MacOS/siril",
                "/usr/bin/siril", shutil.which("siril") or ""]),
]
for prog, paths in _BINS:
    found = next((p for p in paths if p and shutil.os.path.exists(p)), None)
    _chk(f"bin: {prog}", found is not None,
         found if found else "nicht gefunden (optional — nur für echte Daten nötig)")

# --- KI-Zweitlesung des Berichts (optional): staerkerer Rechner im Heimnetz, sonst lokal; dort Opencode vor Ollama ---
# Konfiguration (Rechner, Adressen) steht im Schluessel "ai" der privaten observer.json (nicht im Repo), s. varphot/aireview.py; VARPHOT_AI=off schaltet ab.
AI_BACKEND = None
try:
    from varphot import detect_backends, choose_backend
    from varphot.aireview import preflight_lines
    _ai = detect_backends()
    AI_BACKEND = choose_backend(_ai)
    for _l, _ok, _d in preflight_lines(_ai):
        _chk(_l, _ok, _d, optional=True)
except Exception as _e:                                    # nie das Notebook blockieren
    _chk("KI-Zweitlesung", False, f"Pruefung fehlgeschlagen: {type(_e).__name__}: {_e}", optional=True)

# --- Kernel-Info ---
_chk("Kernel", True, f"Python {sys.version.split()[0]}  —  {sys.executable}")

# --- HTML-Ausgabe ---
rows = ""
all_ok = True
for label, ok, detail, optional in _checks:
    icon  = "✓" if ok else ("·" if optional else "✗")
    color = "#4caf50" if ok else ("#888" if optional else "#f44336")
    if not ok and not optional:
        all_ok = False
    rows += (
        f'<tr><td style="padding:3px 10px;color:{color};font-weight:bold">{icon}</td>'
        f'<td style="padding:3px 10px">{label}</td>'
        f'<td style="padding:3px 10px;color:#888;font-size:12px">{detail}</td></tr>'
    )

status_color = "#4caf50" if all_ok else "#f44336"
status_text  = "Alle Vorbedingungen erfüllt — Notebook kann gestartet werden." \
               if all_ok else "⚠ Mindestens eine Vorbedingung fehlt (s. ✗-Zeilen)."

display(HTML(f"""
<table style="border-collapse:collapse;font-family:monospace;font-size:13px;margin:6px 0">
  <tr style="background:#1e1e2e;color:#cdd6f4">
    <th colspan="3" style="padding:6px 14px;text-align:left">
      Preflight-Check — Notebook 01 Kalibrierung
    </th>
  </tr>
  {rows}
  <tr><td colspan="3" style="padding:6px 10px;color:{status_color};font-weight:bold">
    {status_text}
  </td></tr>
</table>
"""))

## Imports

Lädt die benötigten Bibliotheken: `ipyfilechooser` für den Ordner-Picker,
`ipywidgets` für die Text-Eingabefelder, `astropy.io.fits` für den FITS-Header-Check.

In [ ]:
from pathlib import Path
from ipyfilechooser import FileChooser
import ipywidgets as widgets
from IPython.display import display
from astropy.io import fits
import json

## 0 — Beobachter, Standort und Kamera

Wird **einmal** ausgefüllt und gespeichert in `PHOTOMETRIE_PY_DATA/observer.json` neben dem Repo (Nextcloud; sonst `~/.config/varphot/`), **nicht** im Repo — das Repo ist öffentlich.
Danach erscheinen die Werte in jedem Notebook automatisch vorbelegt.

| Angabe | wofür |
|---|---|
| **Standort** (Breite, Länge Ost +, Höhe) | BJD_TDB (Lichtlaufzeit) und Luftmasse |
| **AAVSO-Beobachtercode** | AAVSO-Export (Extended Format) |
| **Kamera** (Gain, Ausleserauschen, Dunkelstrom, Linearität) | Fehlerbudget und AAVSO-Formular „Detector Details" |

Standort wählen oder über *Neuer Standort* eintragen, dann **Speichern**. Die Testdaten-Standorte (LCO McDonald, Whipple) sind mit dabei.


In [ ]:
from dataclasses import replace
from varphot import PRESETS
from varphot.observer import (Camera, Observer, Site, all_sites, camera_label, config_path,
                              load_config, save_config, upsert_camera, upsert_site)

_obs, _own_sites = load_config()
if _obs.camera.model and all((c.model, c.gain_setting) != (_obs.camera.model, _obs.camera.gain_setting) for c in _obs.cameras):
    _obs.cameras.append(_obs.camera)           # aeltere Datei: die einzelne aktive Kamera wird zum ersten Profil
_sites = all_sites(_own_sites)
_by_name = {s.name: s for s in _sites}
_NEW = "➕ Neuer Standort …"

_st = {"description_width": "150px"}
_lay = widgets.Layout(width="520px")
_num = lambda desc, val, **kw: widgets.FloatText(description=desc, value=val if val is not None else 0.0,
                                                 style=_st, layout=_lay, **kw)

w_site   = widgets.Dropdown(description="Standort:", options=[s.name for s in _sites] + [_NEW],
                            value=(_obs.site_name if _obs.site_name in _by_name else
                                   (_sites[0].name if _sites else _NEW)), style=_st, layout=_lay)
w_sname  = widgets.Text(description="Name:", style=_st, layout=_lay)
w_lat    = widgets.BoundedFloatText(description="Breite (° N):", min=-90, max=90, step=0.0001, style=_st, layout=_lay)
w_lon    = widgets.BoundedFloatText(description="Länge (° O, W = −):", min=-180, max=180, step=0.0001, style=_st, layout=_lay)
w_hgt    = widgets.FloatText(description="Höhe (m):", style=_st, layout=_lay)
w_code   = widgets.Text(description="AAVSO-Code:", value=_obs.aavso_code, placeholder="z. B. ABCD", style=_st, layout=_lay)
_CAM_NEW = "➕ Neue Kamera …"
_cam_opts = ([(f"Profil: {camera_label(c)}", ("profil", i)) for i, c in enumerate(_obs.cameras)] +
             [(f"Vorlage: {pr.model}", ("vorlage", k)) for k, pr in PRESETS.items() if k != "eigene"] + [(_CAM_NEW, ("neu", ""))])
_active = next((("profil", i) for i, c in enumerate(_obs.cameras) if (c.model, c.gain_setting) == (_obs.camera.model, _obs.camera.gain_setting)
                and _obs.camera.model), ("neu", ""))
w_camsel = widgets.Dropdown(description="Kamera wählen:", options=_cam_opts, value=_active, style=_st, layout=_lay)
w_cmodel = widgets.Text(description="Kamera-Modell:", value=_obs.camera.model, style=_st, layout=_lay)
w_ctype  = widgets.Dropdown(description="Kamera-Typ:", options=["CMOS", "CCD", "DSLR", "Sonstige"],
                            value=_obs.camera.cam_type if _obs.camera.cam_type in ("CMOS", "CCD", "DSLR", "Sonstige") else "CMOS",
                            style=_st, layout=_lay)
w_gain   = widgets.Text(description="Gain (e⁻/ADU):", value="" if _obs.camera.gain_e_per_adu is None else str(_obs.camera.gain_e_per_adu), placeholder="leer = unbekannt", style=_st, layout=_lay)
w_rn     = widgets.Text(description="Ausleserauschen (e⁻):", value="" if _obs.camera.readout_noise_e is None else str(_obs.camera.readout_noise_e), placeholder="leer = unbekannt", style=_st, layout=_lay)
w_dark   = widgets.Text(description="Dunkelstrom (e⁻/px/s):", value="" if _obs.camera.dark_current_e_per_s is None else str(_obs.camera.dark_current_e_per_s), placeholder="leer = unbekannt", style=_st, layout=_lay)
w_dtemp  = widgets.Text(description="… bei Temperatur (°C):", value="" if _obs.camera.dark_temp_c is None else str(_obs.camera.dark_temp_c), style=_st, layout=_lay)
w_lin    = widgets.Text(description="Linearität:", value=_obs.camera.linearity, placeholder="z. B. 99 % bis 40000 ADU", style=_st, layout=_lay)
w_btn    = widgets.Button(description="Speichern", button_style="success", icon="save")
w_msg    = widgets.HTML()

def _fill_site(*_):
    s = _by_name.get(w_site.value)
    new = s is None
    w_sname.value = "" if new else s.name
    w_lat.value, w_lon.value, w_hgt.value = (0.0, 0.0, 0.0) if new else (s.lat_deg, s.lon_deg, s.height_m)
    for w in (w_sname, w_lat, w_lon, w_hgt):
        w.disabled = not new and s in [b for b in _sites if b not in _own_sites]   # Testdaten-Standorte sind fest
w_site.observe(_fill_site, names="value")
_fill_site()

def _fill_cam(*_):
    kind, ref = w_camsel.value
    if kind == "profil":
        c = _obs.cameras[ref]
        vals = (c.model, c.cam_type, c.gain_e_per_adu, c.readout_noise_e, c.dark_current_e_per_s, c.dark_temp_c, c.linearity)
    elif kind == "vorlage":                      # Vorlage liefert nur Modell/Typ; Messwerte kommen aus NB07
        pr = PRESETS[ref]
        vals = (pr.model, pr.cam_type, None, None, None, None, "")
    else:
        vals = ("", "CMOS", None, None, None, None, "")
    m, t, g, r, d, dt, lin = vals
    w_cmodel.value = m
    w_ctype.value = "DSLR" if t.startswith("DSLR") else (t if t in w_ctype.options else "CMOS")
    for w, v in ((w_gain, g), (w_rn, r), (w_dark, d), (w_dtemp, dt)):
        w.value = "" if v is None else str(v)
    w_lin.value = lin
w_camsel.observe(_fill_cam, names="value")
_fill_cam()

def _base_camera():
    """Ausgangsprofil fuer das Speichern: gewaehltes Profil bzw. Vorlage; bewahrt die NB07-Felder (Bayer-Muster, Bittiefe, Maximalwert …)."""
    kind, ref = w_camsel.value
    if kind == "profil":
        return _obs.cameras[ref]
    if kind == "vorlage":
        pr = PRESETS[ref]
        return Camera(bayer_pattern=pr.bayer_pattern, bit_depth=pr.bit_depth)
    return Camera()

def _opt(txt):
    txt = txt.strip().replace(",", ".")
    return float(txt) if txt else None

def _save(_):
    global _own_sites, _sites, _by_name
    try:
        site = Site(w_sname.value, w_lat.value, w_lon.value, w_hgt.value)
        cam = replace(_base_camera(), model=w_cmodel.value.strip(), cam_type=w_ctype.value, gain_e_per_adu=_opt(w_gain.value),
                      readout_noise_e=_opt(w_rn.value), dark_current_e_per_s=_opt(w_dark.value), dark_temp_c=_opt(w_dtemp.value),
                      linearity=w_lin.value.strip())
        obs = Observer(w_code.value, site.name, cam, upsert_camera(_obs.cameras, cam) if cam.model else list(_obs.cameras))
        if site.name not in {s.name for s in _sites} or site.name in {s.name for s in _own_sites}:
            _own_sites = upsert_site(_own_sites, site)
        path = save_config(obs, _own_sites)
    except ValueError as e:
        w_msg.value = f'<span style="color:#f44336">✗ {e}</span>'
        return
    _sites = all_sites(_own_sites); _by_name = {s.name: s for s in _sites}
    w_site.options = [s.name for s in _sites] + [_NEW]; w_site.value = site.name
    _o2, _ = load_config(); _obs.cameras[:] = _o2.cameras           # Profilliste aktuell halten
    w_msg.value = f'<span style="color:#4caf50">✓ gespeichert: {path}</span>'
w_btn.on_click(_save)

display(widgets.VBox([
    widgets.HTML("<b>Standort</b>"), w_site, w_sname, w_lat, w_lon, w_hgt,
    widgets.HTML("<b>AAVSO</b>"), w_code,
    widgets.HTML("<b>Kamera (Detector Details)</b>"), w_camsel, w_cmodel, w_ctype, w_gain, w_rn, w_dark, w_dtemp, w_lin,
    w_btn, w_msg,
]))


### Beobachter übernehmen

Zelle ausführen **nachdem** oben gespeichert wurde. Zeigt, was an die späteren Notebooks weitergegeben wird, und rechnet als Gegenprobe die **Höhe des Polarsterns** aus (muss ≈ der Breite sein).

In [ ]:
OBSERVER, _own_sites = load_config()
SITE = next((s for s in all_sites(_own_sites) if s.name == OBSERVER.site_name), None)
if SITE is None:
    raise RuntimeError("Bitte oben einen Standort wählen/anlegen und auf *Speichern* klicken.")

print(f"Standort : {SITE.name}")
print(f"           {SITE.lat_deg:.4f}° N, {SITE.lon_deg:.4f}° O, {SITE.height_m:.0f} m")
print(f"AAVSO    : {OBSERVER.aavso_code or '— fehlt (nur zum Einreichen nötig) —'}")
c = OBSERVER.camera
print(f"Kamera   : {c.model or '—'} ({c.cam_type}), Gain {c.gain_e_per_adu or '?'} e-/ADU, "
      f"RN {c.readout_noise_e or '?'} e-, Dunkelstrom {c.dark_current_e_per_s or '?'} e-/px/s bei {c.dark_temp_c if c.dark_temp_c is not None else '?'} °C")

# Gegenprobe: Polaris-Höhe ≈ geografische Breite (J2000-Position fest, kein Netz nötig)
from astropy.coordinates import AltAz, SkyCoord
from astropy.time import Time
_t = Time.now()
_aa = SkyCoord(37.954561, 89.264109, unit="deg").transform_to(AltAz(obstime=_t, location=SITE.earth_location))
print(f"Polaris  : Höhe {_aa.alt.deg:.2f}° (Breite {SITE.lat_deg:.2f}°) — Abweichung bis ~1° ist normal (Pol-Abstand 0,7°)")


## 1 — Siril-Workspace wählen

Navigiere zum Ordner, den du in Siril als Arbeitsverzeichnis eingestellt hast
(Siril → Datei → Einstellungen → Verzeichnisse). Die Unterordner `lights/`, `darks/`,
`flats/` und `biases/` werden automatisch erkannt.

In [ ]:
# Siril-Workspace-Ordner wählen
ws_chooser = FileChooser(
    path=str(Path.home()),
    title="Siril-Workspace (Ordner mit lights/, darks/, flats/, biases/)",
    show_only_dirs=True,
    show_hidden=False,
)
display(ws_chooser)

## 2 — Ordnerstruktur erkennen und anzeigen

Zelle ausführen **nachdem** oben ein Ordner gewählt wurde.

In [ ]:
# Siril-Vorgabenamen je Kategorie (Groß-/Kleinschreibung)
_SIRIL_DIRS = {
    "lights":  ["lights", "Light", "Lights", "light", "LIGHTS"],
    "darks":   ["darks",  "Dark",  "Darks",  "dark",  "DARKS"],
    "flats":   ["flats",  "Flat",  "Flats",  "flat",  "FLATS"],
    "biases":  ["biases", "Bias",  "bias",   "Biases","BIAS", "bias_frames"],
    "process": ["process","Process","PROCESS"],
}
_FITS_GLOB = ["*.fits", "*.fit", "*.fts", "*.FITS", "*.FIT", "*.FTS"]

def _fits_in(folder):
    files = []
    for pat in _FITS_GLOB:
        files.extend(folder.glob(pat))
    return sorted(set(files))

def detect_siril_workspace(workspace):
    ws = Path(workspace)
    result = {}
    for key, names in _SIRIL_DIRS.items():
        for name in names:
            p = ws / name
            if p.is_dir():
                files = _fits_in(p)
                result[key] = {"path": p, "count": len(files), "files": files}
                break
        else:
            result[key] = {"path": None, "count": 0, "files": []}
    return result

if not ws_chooser.selected:
    raise RuntimeError("Bitte zuerst oben einen Ordner wählen und dann diese Zelle erneut ausführen.")

WORKSPACE = Path(ws_chooser.selected)
dirs = detect_siril_workspace(WORKSPACE)

print(f"Workspace: {WORKSPACE}\n")
print(f"{'Kategorie':<10} {'Ordner':<40} {'FITS'}")  
print("-" * 60)
for key, info in dirs.items():
    folder_str = str(info["path"].relative_to(WORKSPACE)) if info["path"] else "— nicht gefunden —"
    print(f"{key:<10} {folder_str:<40} {info['count']}")

## 3 — Pfade manuell korrigieren (optional)

Falls die automatische Erkennung einen Ordner nicht gefunden hat oder
du von der Siril-Standardstruktur abweichst, hier den Pfad eintragen.

In [ ]:
def _path_or(detected, manual):
    """Gibt den manuellen Pfad zurück wenn angegeben, sonst den automatisch erkannten."""
    if manual.strip():
        p = Path(manual.strip())
        return p if p.is_dir() else None
    return detected

style = {"description_width": "80px"}
layout = widgets.Layout(width="500px")

w_lights  = widgets.Text(description="lights:",  placeholder="leer lassen = auto", style=style, layout=layout)
w_darks   = widgets.Text(description="darks:",   placeholder="leer lassen = auto", style=style, layout=layout)
w_flats   = widgets.Text(description="flats:",   placeholder="leer lassen = auto", style=style, layout=layout)
w_biases  = widgets.Text(description="biases:",  placeholder="leer lassen = auto", style=style, layout=layout)
w_cal_out = widgets.Text(description="cal-out:", placeholder="Ausgabe kalibrierter FITS (leer = process/)",
                         style=style, layout=layout)

# Vorbefüllen mit erkannten Pfaden
for w, key in [(w_lights, "lights"), (w_darks, "darks"),
               (w_flats, "flats"), (w_biases, "biases")]:
    if dirs[key]["path"]:
        w.value = str(dirs[key]["path"])
if dirs["process"]["path"]:
    w_cal_out.value = str(dirs["process"]["path"])

display(widgets.VBox([
    widgets.HTML("<b>Pfade (bei Bedarf anpassen):</b>"),
    w_lights, w_darks, w_flats, w_biases,
    widgets.HTML("<b>Ausgabe kalibrierte FITS:</b>"),
    w_cal_out,
]))

## 4 — Pfade übernehmen und FITS-Header prüfen

Zelle ausführen **nachdem** oben alle Pfade gesetzt sind.

In [ ]:
DIR_LIGHTS = _path_or(dirs["lights"]["path"],  w_lights.value)
DIR_DARKS  = _path_or(dirs["darks"]["path"],   w_darks.value)
DIR_FLATS  = _path_or(dirs["flats"]["path"],   w_flats.value)
DIR_BIASES = _path_or(dirs["biases"]["path"],  w_biases.value)
DIR_CAL    = Path(w_cal_out.value.strip()) if w_cal_out.value.strip() \
             else WORKSPACE / "process"
DIR_CAL.mkdir(parents=True, exist_ok=True)

print("Aktive Pfade:")
for label, p in [("lights", DIR_LIGHTS), ("darks", DIR_DARKS),
                 ("flats", DIR_FLATS), ("biases", DIR_BIASES), ("cal-out", DIR_CAL)]:
    status = "✓" if (p and p.exists()) else "✗ FEHLT"
    print(f"  {label:<10} {status}  {p}")

# FITS-Header des ersten Light-Frames prüfen
lights = _fits_in(DIR_LIGHTS) if DIR_LIGHTS else []
print(f"\n{len(lights)} Light-Frames gefunden.")
if lights:
    with fits.open(lights[0]) as hdul:
        h = hdul[0].header
    chk_keys = ["DATE-OBS", "EXPTIME", "FILTER", "OBJECT",
                "XBINNING", "GAIN", "CCD-TEMP", "SITELAT", "SITELONG"]
    print("\nHeader-Check (erster Light-Frame):")
    for k in chk_keys:
        val = h.get(k, "— fehlt —")
        flag = "  " if k in h else " !"
        print(f"  {flag} {k:<12} {val}")

## 5 — Kalibrierung

**Option A — Siril (empfohlen)**: Kalibrierung in Siril durchführen, Ergebnis landet in `process/`.
Dann oben unter *cal-out* den `process/`-Pfad eintragen und Schritt 4 erneut ausführen.

**Option B — ccdproc (Python)**: Zelle unten aktivieren und anpassen.

In [ ]:
# Option B: ccdproc — noch nicht implementiert
# from ccdproc import ImageFileCollection, Combiner, ccd_process
# ...
print("Kalibrierung: Siril-Ergebnis verwenden oder ccdproc-Block aktivieren.")

## 6 — Konfiguration für Notebook 02 speichern

In [ ]:
cfg_path = WORKSPACE / "session_config.json"
session_cfg = {
    "workspace":   str(WORKSPACE),
    "dir_lights":  str(DIR_LIGHTS)  if DIR_LIGHTS  else None,
    "dir_darks":   str(DIR_DARKS)   if DIR_DARKS   else None,
    "dir_flats":   str(DIR_FLATS)   if DIR_FLATS   else None,
    "dir_biases":  str(DIR_BIASES)  if DIR_BIASES  else None,
    "dir_cal":     str(DIR_CAL),
    "n_lights":    len(lights),
    "observer": {
        "aavso_code": OBSERVER.aavso_code,
        "site": {"name": SITE.name, "lat_deg": SITE.lat_deg, "lon_deg": SITE.lon_deg, "height_m": SITE.height_m},
    },
}
cfg_path.write_text(json.dumps(session_cfg, indent=2, ensure_ascii=False))
print(f"Konfiguration gespeichert: {cfg_path}")
print("→ Notebook 02 liest diese Datei automatisch.")